
# Дообучение ResNet-18 по нарезанным видео светофоров


## 1. Подготавливка окружения:

In [ ]:
from dataclasses import dataclass, field, asdict
from pathlib import Path
import json
import random
import shutil
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from ultralytics import YOLO
from tqdm.auto import tqdm

@dataclass
class TrainConfig:
    ROOT_DIR: str = "."
    OUT_DIR: str = "tl_custom_train"
    YOLO_WEIGHTS: str = "yolov8s.pt"
    VIDEO_LABELS: dict = field(default_factory=lambda: { # Соответствие исходных видео целевым классам.
        "RedLight_Dark.mp4": "red",
        "RedLight_Light.mp4": "red",
        "GreenLight1_Dark.mp4": "green",
        "GreenLight2_Dark.mp4": "green",
        "GreenLight_Light.mp4": "green",
    })

    TL_CLASS_ID: int = 9 # Идентификатор класса traffic light в COCO-разметке YOLO. (стандарт светофора 9)
    YOLO_CONF: float = 0.10
    MAX_SEARCH_FRAMES: int = 45  
    HOLD_FRAMES: int = 3     
    CROP_PAD_RATIO: float = 0.18 
    MIN_AREA: int = 50 
    MAX_AREA: int = 12000
    MAX_CENTER_Y_RATIO: float = 0.80
    INPUT_SIZE: int = 96
    VAL_RATIO: float = 0.20
    SPLIT_MODE: str = "by_frame"
    SEED: int = 42
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 0
    STAGE1_EPOCHS: int = 3          # обучаем только классификационный слой
    STAGE2_EPOCHS: int = 10         # дообучаем всю сеть
    LR_STAGE1: float = 1e-3
    LR_STAGE2: float = 1e-4
    WEIGHT_DECAY: float = 1e-4
    EXPORT_FOR_CASCADE: bool = True
    CASCADE_WEIGHTS_NAME: str = "resnet18_tl_state_dict.pt"
    BEST_WEIGHTS_NAME: str = "resnet18_tl_custom_state_dict.pt"
    META_NAME: str = "resnet18_tl_custom_meta.json"

cfg = TrainConfig()
CLASS_TO_IDX = {"red": 0, "green": 1}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

ROOT = Path(cfg.ROOT_DIR)
OUT = ROOT / cfg.OUT_DIR
ROI_DIR = OUT / "roi"
RED_DIR = ROI_DIR / "red"
GREEN_DIR = ROI_DIR / "green"
TRACK_DIR = OUT / "tracks"
SPLIT_DIR = OUT / "splits"
WEIGHTS_PATH = OUT / cfg.BEST_WEIGHTS_NAME
META_PATH = OUT / cfg.META_NAME

for p in [OUT, ROI_DIR, RED_DIR, GREEN_DIR, TRACK_DIR, SPLIT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(cfg), ensure_ascii=False, indent=2))



## 2. Вспомогательные функции для детекции, трекинга и вырезания ROI:

In [ ]:

def iou_xyxy(a, b): # Вычисляет IoU между двумя ограничивающими рамками
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(1.0, ax2 - ax1) * max(1.0, ay2 - ay1)
    area_b = max(1.0, bx2 - bx1) * max(1.0, by2 - by1)
    return inter / (area_a + area_b - inter + 1e-6)

def expand_box(box, width, height, pad_ratio=0.18): # Расширяет рамку на заданный процент от её размера, не выходя за границы изображения
    x1, y1, x2, y2 = box
    bw = x2 - x1
    bh = y2 - y1
    px = int(round(bw * pad_ratio))
    py = int(round(bh * pad_ratio))
    nx1 = max(0, int(round(x1 - px)))
    ny1 = max(0, int(round(y1 - py)))
    nx2 = min(width, int(round(x2 + px)))
    ny2 = min(height, int(round(y2 + py)))
    return nx1, ny1, nx2, ny2

def candidate_score(box, conf, frame_shape, cfg): # Вычисляет эвристическую оценку качества кандидата
    h, w = frame_shape[:2]
    x1, y1, x2, y2 = box
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    area = bw * bh
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    if area < cfg.MIN_AREA or area > cfg.MAX_AREA:
        return None
    if cy > cfg.MAX_CENTER_Y_RATIO * h:
        return None
    x_center_score = abs(cx - w / 2.0) / (w / 2.0 + 1e-6)
    y_score = cy / (h + 1e-6)
    area_score = 1.0 / np.sqrt(area)
    score = 2.0 * x_center_score + 1.0 * y_score + 0.5 * area_score - 0.2 * conf
    return float(score)

def collect_candidates(frame, model, cfg): # Собирает кандидатов на рамки светофора в кадре с помощью модели YOLO и оценивает их эвристической функцией
    result = model.predict(frame, conf=cfg.YOLO_CONF, verbose=False)[0]
    cands = []
    for b in result.boxes:
        if int(b.cls.item()) != cfg.TL_CLASS_ID:
            continue
        x1, y1, x2, y2 = map(float, b.xyxy[0].tolist())
        conf = float(b.conf.item())
        box = (x1, y1, x2, y2)
        score = candidate_score(box, conf, frame.shape, cfg)
        if score is None:
            continue
        cands.append({
            "bbox": box,
            "conf": conf,
            "score": score,
        })
    cands.sort(key=lambda x: x["score"])
    return cands

def find_anchor_bbox(video_path, model, cfg): # Находит начальную рамку для отслеживания
    cap = cv2.VideoCapture(str(video_path))
    anchor = None
    search_meta = []
    for idx in range(cfg.MAX_SEARCH_FRAMES):
        ok, frame = cap.read()
        if not ok:
            break
        cands = collect_candidates(frame, model, cfg)
        search_meta.append({"frame_idx": idx, "n_candidates": len(cands)})
        if cands:
            anchor = cands[0]["bbox"]
            break
    cap.release()
    return anchor, search_meta

def select_tracked_bbox(cands, prev_bbox): # Выбирает лучшую рамку для текущего кадра на основе IoU с предыдущей рамкой и эвристической оценки
    if not cands:
        return None
    if prev_bbox is None:
        return cands[0]
    best = None
    best_score = -1e9
    for cand in cands:
        bb = cand["bbox"]
        iou = iou_xyxy(prev_bbox, bb)
        track_score = 1.8 * iou + 0.4 * cand["conf"] - 0.15 * cand["score"]
        if track_score > best_score:
            best_score = track_score
            best = cand.copy()
            best["iou_prev"] = float(iou)
            best["track_score"] = float(track_score)
    return best

def extract_rois_from_video(video_name, label, model, cfg): # Извлекает ROI светофора из видео, выполняя детекцию и простое отслеживание, сохраняет их в каталог и возвращает список метаданных
    video_path = ROOT / video_name
    label_dir = RED_DIR if label == "red" else GREEN_DIR
    track_json_path = TRACK_DIR / f"{Path(video_name).stem}_track.json"
    anchor_bbox, search_meta = find_anchor_bbox(video_path, model, cfg)
    cap = cv2.VideoCapture(str(video_path))
    frame_idx = 0
    prev_bbox = anchor_bbox
    hold_left = 0
    rows = []
    pbar = tqdm(total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0), desc=Path(video_name).stem)
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        h, w = frame.shape[:2]
        cands = collect_candidates(frame, model, cfg)
        picked = select_tracked_bbox(cands, prev_bbox)
        active_bbox = None
        mode = "none"
        conf = None
        score = None
        iou_prev = None
        track_score = None
        if picked is not None:
            active_bbox = picked["bbox"]
            prev_bbox = active_bbox
            hold_left = cfg.HOLD_FRAMES
            mode = "detected"
            conf = picked.get("conf")
            score = picked.get("score")
            iou_prev = picked.get("iou_prev")
            track_score = picked.get("track_score")
        elif prev_bbox is not None and hold_left > 0:
            active_bbox = prev_bbox
            hold_left -= 1
            mode = "hold"
        else:
            prev_bbox = None
            active_bbox = None
            mode = "miss"

        crop_path = None
        if active_bbox is not None:
            ex1, ey1, ex2, ey2 = expand_box(active_bbox, w, h, cfg.CROP_PAD_RATIO)
            roi = frame[ey1:ey2, ex1:ex2].copy()
            if roi.size > 0:
                crop_name = f"{Path(video_name).stem}_f{frame_idx:06d}.png"
                crop_path = label_dir / crop_name
                cv2.imwrite(str(crop_path), roi)

        rows.append({
            "video": video_name,
            "label": label,
            "frame_idx": frame_idx,
            "mode": mode,
            "bbox_x1": float(active_bbox[0]) if active_bbox is not None else None,
            "bbox_y1": float(active_bbox[1]) if active_bbox is not None else None,
            "bbox_x2": float(active_bbox[2]) if active_bbox is not None else None,
            "bbox_y2": float(active_bbox[3]) if active_bbox is not None else None,
            "det_conf": conf,
            "heuristic_score": score,
            "iou_prev": iou_prev,
            "track_score": track_score,
            "crop_path": str(crop_path) if crop_path is not None else None,
        })
        frame_idx += 1
        pbar.update(1)
    pbar.close()
    cap.release()
    with open(track_json_path, "w", encoding="utf-8") as f:
        json.dump({
            "video": video_name,
            "label": label,
            "anchor_bbox": list(map(float, anchor_bbox)),
            "search_meta": search_meta,
            "track": rows,
        }, f, ensure_ascii=False, indent=2)
    return rows

## 3. Извлечение ROI из всех нарезанных видео

In [ ]:
for p in [RED_DIR, GREEN_DIR, TRACK_DIR]: # При необходимости можно очистить результаты прошлого прогона
    if p.exists():
        for child in p.iterdir():
            if child.is_file():
                child.unlink()
            elif child.is_dir():
                shutil.rmtree(child)
        p.mkdir(parents=True, exist_ok=True)
model_yolo = YOLO(cfg.YOLO_WEIGHTS)
all_rows = []
for video_name, label in cfg.VIDEO_LABELS.items():
    rows = extract_rois_from_video(video_name, label, model_yolo, cfg)
    all_rows.extend(rows)
roi_df = pd.DataFrame(all_rows)
roi_df.to_csv(OUT / "roi_metadata.csv", index=False, encoding="utf-8-sig")
saved_df = roi_df.dropna(subset=["crop_path"]).copy()
saved_df["target"] = saved_df["label"].map(CLASS_TO_IDX)
print("Всего строк трекинга:", len(roi_df))
print("Всего сохранённых ROI:", len(saved_df))
display(saved_df.groupby(["label", "video"]).size().reset_index(name="n_rois"))


## 4. Разбиение на train / val


In [ ]:
def make_splits(df, cfg):
    rng = np.random.default_rng(cfg.SEED)
    df = df.sample(frac=1.0, random_state=cfg.SEED).reset_index(drop=True).copy()
    if cfg.SPLIT_MODE == "by_frame":
        train_parts = []
        val_parts = []
        for label in sorted(df["label"].unique()):
            part = df[df["label"] == label].copy().reset_index(drop=True)
            idx = np.arange(len(part))
            rng.shuffle(idx)
            n_val = max(1, int(round(len(part) * cfg.VAL_RATIO)))
            val_idx = idx[:n_val]
            train_idx = idx[n_val:]
            if len(train_idx) == 0:
                train_idx = idx[1:]
                val_idx = idx[:1]
            train_parts.append(part.iloc[train_idx])
            val_parts.append(part.iloc[val_idx])
        train_df = pd.concat(train_parts, ignore_index=True)
        val_df = pd.concat(val_parts, ignore_index=True)
    elif cfg.SPLIT_MODE == "by_video":
        train_rows = []
        val_rows = []
        for label in sorted(df["label"].unique()):
            part = df[df["label"] == label].copy()
            videos = sorted(part["video"].unique())
            rng.shuffle(videos)
            n_val_videos = 1 if len(videos) >= 2 else 0
            val_videos = set(videos[:n_val_videos])
            val_rows.append(part[part["video"].isin(val_videos)])
            train_rows.append(part[~part["video"].isin(val_videos)])
        train_df = pd.concat(train_rows, ignore_index=True)
        val_df = pd.concat(val_rows, ignore_index=True)
    else:
        raise ValueError(f"Неизвестный режим разбиения: {cfg.SPLIT_MODE}")
    train_df = train_df.sample(frac=1.0, random_state=cfg.SEED).reset_index(drop=True)
    val_df = val_df.sample(frac=1.0, random_state=cfg.SEED).reset_index(drop=True)
    return train_df, val_df

train_df, val_df = make_splits(saved_df, cfg)
train_df.to_csv(SPLIT_DIR / "train.csv", index=False, encoding="utf-8-sig")
val_df.to_csv(SPLIT_DIR / "val.csv", index=False, encoding="utf-8-sig")
print("train:", len(train_df), "val:", len(val_df))
display(train_df.groupby(["label", "video"]).size().reset_index(name="n_train"))
display(val_df.groupby(["label", "video"]).size().reset_index(name="n_val"))


## 5. Dataset и DataLoader

In [ ]:

train_tf = transforms.Compose([
    transforms.Resize((cfg.INPUT_SIZE, cfg.INPUT_SIZE)),
    transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.20, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize((cfg.INPUT_SIZE, cfg.INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class TLFrameDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["crop_path"]).convert("RGB")
        x = self.transform(img) if self.transform is not None else img
        y = int(row["target"])
        return x, torch.tensor(y, dtype=torch.long), row["video"], int(row["frame_idx"])

train_ds = TLFrameDataset(train_df, transform=train_tf)
val_ds = TLFrameDataset(val_df, transform=val_tf)
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=cfg.NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS)

class_counts = train_df["target"].value_counts().sort_index().reindex([0, 1], fill_value=0)
class_weights = torch.tensor(
    [len(train_df) / max(1, class_counts.get(i, 0)) for i in [0, 1]],
    dtype=torch.float32,
    device=device,
)
print("class_counts:", class_counts.to_dict())
print("class_weights:", class_weights.detach().cpu().tolist())

## 6. Модель и обучение

In [ ]:

def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model.to(device)

def set_backbone_trainable(model, train_backbone: bool):
    for name, p in model.named_parameters():
        p.requires_grad = train_backbone
    for p in model.fc.parameters():
        p.requires_grad = True

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y, _, _ in loader:
        x = x.to(device)
        y = y.to(device)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_train):
            logits = model(x)
            loss = criterion(logits, y)
            if is_train:
                loss.backward()
                optimizer.step()
        total_loss += float(loss.item()) * x.size(0)
        pred = torch.argmax(logits, dim=1)
        correct += int((pred == y).sum().item())
        total += int(x.size(0))
    return {
        "loss": total_loss / max(1, total),
        "acc": correct / max(1, total),
        "n": total,
    }

def evaluate_df(model, df, transform):
    model.eval()
    rows = []
    with torch.no_grad():
        for _, r in tqdm(df.iterrows(), total=len(df), desc="eval_df"):
            img = Image.open(r["crop_path"]).convert("RGB")
            x = transform(img).unsqueeze(0).to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()
            pred = int(np.argmax(probs))
            rows.append({
                "video": r["video"],
                "frame_idx": int(r["frame_idx"]),
                "label": r["label"],
                "target": int(r["target"]),
                "pred": pred,
                "pred_label": IDX_TO_CLASS[pred],
                "p_red": float(probs[0]),
                "p_green": float(probs[1]),
                "crop_path": r["crop_path"],
            })
    pred_df = pd.DataFrame(rows)
    acc = float((pred_df["pred"] == pred_df["target"]).mean()) if len(pred_df) else float("nan")
    return pred_df, acc


## 7.Двухэтапное дообучение классификатора ResNet-18

In [ ]:

model = build_model()
criterion = nn.CrossEntropyLoss(weight=class_weights)
history = []
best_state = None
best_val_acc = -1.0
best_val_loss = 1e9
set_backbone_trainable(model, train_backbone=False)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg.LR_STAGE1, weight_decay=cfg.WEIGHT_DECAY)

for epoch in range(1, cfg.STAGE1_EPOCHS + 1):
    tr = run_epoch(model, train_loader, criterion, optimizer=optimizer)
    va = run_epoch(model, val_loader, criterion, optimizer=None)
    row = {"stage": 1, "epoch": epoch, **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in va.items()}}
    history.append(row)
    print(row)
    if (va["acc"] > best_val_acc) or (va["acc"] == best_val_acc and va["loss"] < best_val_loss):
        best_val_acc = va["acc"]
        best_val_loss = va["loss"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

set_backbone_trainable(model, train_backbone=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR_STAGE2, weight_decay=cfg.WEIGHT_DECAY)

for epoch in range(1, cfg.STAGE2_EPOCHS + 1):
    tr = run_epoch(model, train_loader, criterion, optimizer=optimizer)
    va = run_epoch(model, val_loader, criterion, optimizer=None)
    row = {"stage": 2, "epoch": epoch, **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in va.items()}}
    history.append(row)
    print(row)
    if (va["acc"] > best_val_acc) or (va["acc"] == best_val_acc and va["loss"] < best_val_loss):
        best_val_acc = va["acc"]
        best_val_loss = va["loss"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

history_df = pd.DataFrame(history)
history_df.to_csv(OUT / "train_history.csv", index=False, encoding="utf-8-sig")
display(history_df)

if best_state is None:
    raise RuntimeError("best_state не был сохранён")
torch.save(best_state, WEIGHTS_PATH)
print("Сохранены лучшие веса:", WEIGHTS_PATH)
print("best_val_acc:", best_val_acc, "best_val_loss:", best_val_loss)


## 8. Оценка лучшей модели

In [ ]:

best_model = build_model()
best_model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
best_model.eval()
train_pred_df, train_acc = evaluate_df(best_model, train_df, val_tf)
val_pred_df, val_acc = evaluate_df(best_model, val_df, val_tf)
all_pred_df, all_acc = evaluate_df(best_model, saved_df, val_tf)
train_pred_df.to_csv(OUT / "train_predictions.csv", index=False, encoding="utf-8-sig")
val_pred_df.to_csv(OUT / "val_predictions.csv", index=False, encoding="utf-8-sig")
all_pred_df.to_csv(OUT / "all_predictions.csv", index=False, encoding="utf-8-sig")
print("train_acc:", train_acc)
print("val_acc:", val_acc)
print("all_acc:", all_acc)
display(pd.crosstab(val_pred_df["label"], val_pred_df["pred_label"], margins=True))
display(
    all_pred_df.groupby(["label", "video"])
    .apply(lambda x: pd.Series({
        "n": len(x),
        "acc": float((x["pred"] == x["target"]).mean()),
        "mean_p_red": float(x["p_red"].mean()),
        "mean_p_green": float(x["p_green"].mean()),
    }))
    .reset_index()
)


## 9. Экспорт весов и метаданных для каскада

In [ ]:

meta = {
    "class_to_idx": CLASS_TO_IDX,
    "idx_to_class": IDX_TO_CLASS,
    "input_size": cfg.INPUT_SIZE,
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "weights_path": str(WEIGHTS_PATH.name),
    "notes": "0=red, 1=green",
}

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

if cfg.EXPORT_FOR_CASCADE:
    cascade_path = OUT / cfg.CASCADE_WEIGHTS_NAME
    shutil.copy2(WEIGHTS_PATH, cascade_path)
    print("Копия весов для каскада сохранена в:", cascade_path)

print("meta.json:", META_PATH)
print(json.dumps(meta, ensure_ascii=False, indent=2))
